In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import config.ConnectionConfig as cc
cc.setupEnvironment()

spark = cc.startLocalCluster("DIM_VEHICLE",4) # 4 partitions, idk why
spark.getActiveSession()

25/05/09 12:40:40 WARN Utils: Your hostname, 4L3KS-comp resolves to a loopback address: 127.0.1.1; using 10.140.99.119 instead (on interface wlp2s0)
25/05/09 12:40:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/aleks/Downloads/bigtools/spark-3.5.4-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/aleks/.ivy2/cache
The jars for the packages stored in: /home/aleks/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.postgresql#postgresql added as a dependency
org.elasticsearch#elasticsearch-spark-30_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f16eda9b-e2fe-4c11-9707-3183826e27b6;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.0 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.9.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.

# EXTRACT

In [2]:
from pyspark.sql.functions import *

cc.set_connectionProfile("default")

df_operational_vehicle = spark.read.format("jdbc")\
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "bike_types") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "biketypeid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 20) \
    .load()
# not needed if we don't do SQL I think.
df_operational_vehicle.createOrReplaceTempView('operational_vehicle')
df_operational_vehicle.show()

+----------+-------------------+
|biketypeid|biketypedescription|
+----------+-------------------+
|         1|          Velo Bike|
|         2|        Velo E-Bike|
|         3|               Step|
|         4|            Scooter|
+----------+-------------------+



# TRANSFORM

In [3]:
# Data is usable in this form. Just renaming it with spark API.
dimType = df_operational_vehicle.withColumnRenamed('biketypeid','bike_type_id').withColumnRenamed( 'biketypedescription', 'bike_type_description')
dimType.show()

+------------+---------------------+
|bike_type_id|bike_type_description|
+------------+---------------------+
|           1|            Velo Bike|
|           2|          Velo E-Bike|
|           3|                 Step|
|           4|              Scooter|
+------------+---------------------+



# LOAD

In [4]:
dimType.write.format("delta").mode("overwrite").saveAsTable("dimVehcileType")
dimType.repartition(1).write.format("parquet").mode("overwrite").saveAsTable("dimVehcileType_parquet")

In [5]:
spark.stop()